Multi-Agent Chatbot with AG2 (AutoGen) for Healthcare


## Objectives

After completing this lab you will be able to:

- Learn how AG2 (AutoGen) enables multi-agent AI systems for complex workflows.
- Explore how AG2 (AutoGen) integrates with LLMs like GPT-4 for dynamic AI-driven conversations.
- Implement agent-to-agent communication for intelligent medical decision-making.
- Develop multiple AI agents that interact and collaborate to handle different healthcare tasks.


----


In [1]:
!pip install autogen==0.7 openai==1.64.0 python-dotenv==1.1.0 gradio==5.15.0 | tail -n 1

### Importing Required Libraries
Import all required libraries:


In [2]:
import warnings

# Suppress autogen and other deprecation/user warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)

from autogen import ConversableAgent, GroupChat, GroupChatManager
from openai import OpenAI
import logging

In [3]:
# Suppress warnings from autogen.oai.client
logging.getLogger("autogen.oai.client").setLevel(logging.ERROR)


# What is AutoGen?

AutoGen is an open-source framework developed by Microsoft that enables developers to orchestrate and optimize AI workflows using multiple AI agents. These agents can collaborate, automate decision-making, and dynamically generate responses in complex problem-solving tasks.

Unlike traditional AI systems that work in isolation, AutoGen enables multiple AI agents (LLMs such as GPT) to interact, exchange information, and refine their outputs, making it more powerful and flexible for various applications.

# Key Features of AutoGen
### 1. Multi-Agent Collaboration

AutoGen allows multiple AI agents to communicate and solve tasks collaboratively. Each agent can have a specific role, such as a problem solver, verifier, or optimizer.

Example:

- One agent generates code, another reviews it, and another tests it.
- A research agent collects information, while another summarizes it.

### 2. Conversational and Task-Oriented AI

AutoGen supports LLM-driven conversations where AI agents engage in multi-turn dialogues to refine answers.

Example:

- A chatbot that consults different AI agents, such as one for legal advice and another for finance.
- A customer support AI that escalates unresolved queries to another AI agent.

### 3. Automated Workflow Generation

You can orchestrate workflows for AI-driven automation, such as AI-assisted programming, research, and document generation.

Example:

- Automating software debugging where AI identifies issues, suggests fixes, and verifies solutions.

### 4. Supports Human-AI Collaboration

AutoGen allows humans to intervene in AI-driven workflows by providing feedback or manually guiding agents when necessary.

Example:

- A research assistant AI drafts a report, but a human expert refines it.








# Comparison: AutoGen vs Traditional AI Agents

<table width="60%" align="left" >
        <thead>
            <tr>
                <th>Feature</th>
                <th>Traditional AI Agents</th>
                <th>AutoGen AI Agents</th>
            </tr>
        </thead>
        <tbody>
            <tr>
                <td>Interactivity</td>
                <td>Works alone</td>
                <td>Collaborates with other agents</td>
            </tr>
            <tr>
                <td>Learning Ability</td>
                <td>Static</td>
                <td>Adaptive & iterative</td>
            </tr>
            <tr>
                <td>Workflow Handling</td>
                <td>Predefined</td>
                <td>Dynamic & evolving</td>
            </tr>
            <tr>
                <td>Human Intervention</td>
                <td>Limited</td>
                <td>Supports human-AI collaboration</td>
            </tr>
        </tbody>
</table>


The OpenAI() function initializes an API client object by calling its constructor, which automatically manages API key retrieval.

The code_execution_config dictionary is set up to signal to other functions that Docker should not be used during code execution, thus ensuring that your code runs directly on the host environment and avoiding Docker-related issues.


In [4]:

# Initialize OpenAI Client (API Key is automatically managed from environment variables or configured in OpenAI settings)
client = OpenAI()

# Disable Docker execution to prevent runtime errors
code_execution_config = {"use_docker": False}

# How AutoMed Works: Multi-Agent AI in Action
When a user interacts with AutoMed, the system does not simply generate a response—it triggers a team of AI agents, each with a specific role in processing the query. These agents collaborate in real-time, cross-validating and optimizing their responses to ensure accuracy and reliability.

For example, consider a patient experiencing persistent headaches and fatigue. Instead of offering generic advice, AutoMed intelligently activates multiple specialized AI agents:

1. **Patient Agent** – Collects user symptoms, medical history, and any ongoing treatments.
2. **Symptom Analyzer Agent** – Evaluates potential conditions such as migraines, dehydration, or anemia based on AI-driven medical knowledge.
3. **Pharmacy Agent** – Suggests remedies, over-the-counter medications, and when to seek professional medical attention.
4.  **Consultation Advisor Agent** (Decides if a doctor visit is needed) – Fetches real-time updates from trusted healthcare medical research papers.

These agents work seamlessly together, analyzing, validating, and optimizing their recommendations to deliver the most relevant, personalized, and up-to-date medical guidance. The collaborative approach ensures that users receive a well-rounded medical consultation experience, similar to interacting with multiple healthcare professionals at once, but through an AI-driven, automated system.


### Why is GPT-4o Used?
GPT-4o is well-suited for:
- Understanding medical symptoms
- Generating detailed, human-like responses
- Providing accurate condition suggestions based on user input


In [5]:
# Sample LLM Configuration (Replace with actual API keys/config if needed)
llm_config = {"config_list": [{"model": "gpt-4", "api_key": None}]}  # Replace with real API key

Now, let's understand the parameters:
- llm_config={"model": "gpt-4o", "api_key": None} : This configures the Large Language Model (LLM) that powers the Diagnosis Agent.
- "model": "gpt-4o" → Specifies that the agent is using OpenAI’s GPT-4o to generate responses.
- "api_key": None → This suggests that the OpenAI API key is not explicitly defined here and is likely set elsewhere in the system configuration- which is provided here by Skills Network. You need an active API key if you plan to run the project outside the CognitiveClass platform.


## What is  ConversableAgent?
- Represents an AI agent that can engage in conversations.
- Each agent has a specific role, defined by its system message.
- Uses LLM (Language Model) configurations to process responses.

 Here, each agent is assigned a specific role, allowing structured communication between AI agents and the user.
- The patient_agent represents the user and is responsible for describing symptoms and requesting medical assistance, but it does not process responses.
- The diagnosis_agent analyzes the symptoms provided by the patient and generates a concise diagnosis in a single response, ensuring clarity and brevity.
- The pharmacy_agent follows up on the diagnosis by recommending medications, but it is restricted to responding only once to prevent unnecessary repetition.
- The consultation_agent plays a critical role in determining whether the patient needs to visit a doctor, providing a final summary of the consultation along with clear next steps. To ensure structured conversation flow, the consultation_agent includes a termination condition by adding "CONSULTATION_COMPLETE" to its response, signaling the end of the consultation session. All agents are configured with the llm_config, which specifies the underlying language model (GPT-4) for processing responses. This structured setup allows for an efficient and logical multi-agent conversation, ensuring that the patient receives a diagnosis, medication recommendations, and a final decision on whether further medical consultation is necessary.


In [6]:

# Step 1: Create AI Agents with Defined Roles
patient_agent = ConversableAgent(
    name="patient", 
    system_message="You describe symptoms and ask for medical help.", 
    llm_config=llm_config
)

diagnosis_agent = ConversableAgent(
    name="diagnosis", 
    system_message="You analyze symptoms and provide a possible diagnosis. Summarize key points in one response.", 
    llm_config=llm_config
)

pharmacy_agent = ConversableAgent(
    name="pharmacy", 
    system_message="You recommend medications based on diagnosis. Only respond once.", 
    llm_config=llm_config
)

consultation_agent = ConversableAgent(
    name="consultation", 
    system_message="You determine if a doctor's visit is required. Provide a final summary with clear next steps. IMPORTANT: End your response with 'CONSULTATION_COMPLETE' to signal the end of the conversation.", 
    llm_config=llm_config
)


## What is GroupChat?

- Manages a structured conversation between multiple AI agents
- Ensures turn-based speaking using speaker selection methods (e.g., round_robin)
- Prevents infinite loops by using max_round

The GroupChat class structures the interaction between multiple AI agents, ensuring a logical conversation flow. Below is a breakdown of each parameter used in the code:

- agents=[diagnosis_agent, pharmacy_agent, consultation_agent]
    - Specifies the AI agents participating in the conversation.
    - The diagnosis agent analyzes symptoms, the pharmacy agent recommends medications, and the consultation agent determines if a doctor's visit is necessary.
    - The patient agent only initiates the conversation and does not actively participate in the group chat.
- messages=[]
    - Initializes the conversation with an empty list of messages.
    - Ensures that no previous data is retained, making each consultation independent.
- max_round=5
    - Limits the conversation to five full cycles through all agents.
    - Prevents infinite loops by restricting the number of exchanges.
    - Ensures the conversation remains efficient and focused.
- speaker_selection_method="round_robin"
    - Controls the order in which agents respond.
    - Uses a "round-robin" approach, meaning each agent takes turns speaking in a structured sequence.
    - Prevents repetition or chaotic interactions, ensuring each agent contributes in a logical order.


In [7]:
# Step 2: Create GroupChat for Structured Interaction
groupchat = GroupChat(
    agents=[diagnosis_agent, pharmacy_agent, consultation_agent],  # Patient only initiates
    messages=[], 
    max_round=5,  # Limits conversation to 5 rounds
    speaker_selection_method="round_robin"  # Ensures structured conversation flow
)

GroupChatManager

- Controls the execution of GroupChat
- Ensures messages flow between agents in an organized manner
- Acts as the conversation coordinator
- Coordinates the multi-agent conversation

It has two parameters. Let's understand both parameters.
- name="manager"
    - Assigns a name to the GroupChatManager.
    - The name "manager" represents the AI-driven entity responsible for orchestrating the conversation between agents.
- groupchat=groupchat
    - Connects the manager to a predefined GroupChat instance.
    - The groupchat parameter contains all agents, including the diagnosis agent, pharmacy agent, and consultation agent.
    - Ensures that all messages and responses follow the structured flow defined in the GroupChat.


In [8]:
# Step 3: Create GroupChatManager to Handle Conversation
manager = GroupChatManager(name="manager", groupchat=groupchat)

In [12]:
import gradio as gr
import io
import contextlib
from autogen import GroupChatManager

# Maintain correct termination configurations 
manager = GroupChatManager(
    name="manager", 
    groupchat=groupchat,
    is_termination_msg=lambda x: "CONSULTATION_COMPLETE" in x.get("content", "") if x.get("content") else False
)

def run_medical_dashboard(user_symptoms):
    if not user_symptoms.strip():
        return (
            """
            <div class='card empty'>
                <div class='empty-title'>⚠️ No Input Detected</div>
                <div class='empty-sub'>Please describe your symptoms to begin diagnostic processing.</div>
            </div>
            """, 
            """
            <div class='log-empty'>System Idle • Awaiting patient signal...</div>
            """
        )

    groupchat.messages = [] 
    terminal_buffer = io.StringIO()

    with contextlib.redirect_stdout(terminal_buffer):
        patient_agent.initiate_chat(
            manager, 
            message=f"I am feeling {user_symptoms}. Can you help?",
            clear_history=True,
            silent=False 
        )
        
    raw_terminal_output = terminal_buffer.getvalue()
    clean_terminal_logs = raw_terminal_output.replace(
        ">>>>>>>> USING AUTO REPLY...", 
        "⚡ AI ROUTER ACTIVE → "
    )

    final_verdict = "Analysis completed. Review system traces below."
    if "consultation (to manager):" in clean_terminal_logs:
        try:
            parts = clean_terminal_logs.split("consultation (to manager):")
            if len(parts) > 1:
                final_verdict = parts[-1].replace("CONSULTATION_COMPLETE", "").strip()
        except Exception:
            pass

    consensus_card = f"""
    <div class="card result-card">
        <div class="card-header">
            🧠 Diagnostic Consensus Engine
        </div>
        <div class="card-body">
            {final_verdict}
        </div>
    </div>
    """

    formatted_logs = f"""
    <div class="card log-card">
        <div class="card-header logs">
            🧾 Agent Execution Trace
        </div>
        <div class="log-body">
            {clean_terminal_logs}
        </div>
    </div>
    """

    return consensus_card, formatted_logs


# =========================
# DESIGN SYSTEM (UPGRADED)
# =========================
custom_dark_css = """
:root {
    --bg: #0b0f14;
    --card: #121823;
    --card2: #0f141d;
    --border: #1f2a3a;
    --text: #e6e6e6;
    --muted: #8b98a5;
    --accent: #4fd1c5;
    --accent2: #60a5fa;
    --danger: #ff5d5d;
    --radius: 10px;
}

/* Base */
.gradio-container, body, html {
    background: radial-gradient(1200px 800px at 20% 0%, #111827 0%, var(--bg) 55%) !important;
    color: var(--text) !important;
    overflow-x: hidden !important;
    font-family: ui-sans-serif, system-ui;
}

/* Layout protection */
p, div, span, pre, code {
    word-break: break-word !important;
}

/* Header */
h1, h2, h3 {
    letter-spacing: 0.5px;
}

/* Inputs */
textarea, input {
    background: var(--card) !important;
    color: var(--text) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--radius) !important;
    padding: 10px !important;
    font-family: ui-monospace, monospace !important;
}

textarea:focus, input:focus {
    border-color: var(--accent) !important;
    box-shadow: 0 0 0 2px rgba(79, 209, 197, 0.15);
}

/* Button */
.action-btn {
    background: linear-gradient(135deg, var(--accent), var(--accent2)) !important;
    color: #0b0f14 !important;
    border: none !important;
    font-weight: 700 !important;
    border-radius: var(--radius) !important;
    padding: 10px 14px !important;
    transition: all 0.2s ease-in-out;
}

.action-btn:hover {
    transform: translateY(-1px);
    filter: brightness(1.1);
}

.action-btn:active {
    transform: scale(0.98);
}

/* Cards */
.card {
    background: linear-gradient(180deg, var(--card), var(--card2));
    border: 1px solid var(--border);
    border-radius: var(--radius);
    padding: 18px;
    box-shadow: 0 10px 30px rgba(0,0,0,0.25);
    margin-bottom: 12px;
}

/* Headers */
.card-header {
    font-weight: 700;
    font-size: 14px;
    color: var(--text);
    border-bottom: 1px solid var(--border);
    padding-bottom: 8px;
    margin-bottom: 12px;
    text-transform: uppercase;
    letter-spacing: 1px;
}

/* Result emphasis */
.result-card .card-body {
    font-size: 15px;
    line-height: 1.6;
    color: var(--text);
}

/* Logs */
.log-card {
    max-height: 520px;
    overflow-y: auto;
}

.log-body {
    font-family: ui-monospace, monospace;
    font-size: 12.5px;
    color: var(--muted);
    white-space: pre-wrap;
}

/* Empty states */
.empty {
    text-align: center;
    padding: 30px;
    border: 1px dashed var(--border);
    color: var(--muted);
}

.empty-title {
    font-size: 16px;
    color: var(--text);
    margin-bottom: 6px;
}

.empty-sub {
    font-size: 13px;
    color: var(--muted);
}

.log-empty {
    color: var(--muted);
    font-family: monospace;
    padding: 10px;
}
"""


with gr.Blocks(css=custom_dark_css) as demo:

    gr.HTML("""
        <div style="text-align:center; padding:18px 0; margin-bottom:18px;">
            <div style="font-size:22px; font-weight:800; color:#e6e6e6;">
                AUTOMED EXPERT ENGINE
            </div>
            <div style="font-size:12px; color:#8b98a5; letter-spacing:1px;">
                Multi-Agent Diagnostic Orchestration System
            </div>
        </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Patient Intake")

            symptom_box = gr.Textbox(
                label="Symptoms",
                placeholder="Describe symptoms clearly (e.g. headache, nausea, fatigue...)",
                lines=6
            )

            submit_action = gr.Button("Run Analysis", elem_classes=["action-btn"])

            gr.HTML("""
                <div class="card" style="font-size:12px; color:#8b98a5;">
                    <b>Active Agents</b><br><br>
                    • diagnosis_agent<br>
                    • pharmacy_agent<br>
                    • consultation_agent
                </div>
            """)

        with gr.Column(scale=2):
            gr.Markdown("### Diagnostic Output")

            consensus_output = gr.HTML("""
                <div class="card empty">
                    Awaiting input to initialize diagnostic graph...
                </div>
            """)

            logs_output = gr.HTML("""
                <div class="card log-empty">
                    System logs will appear here...
                </div>
            """)

    submit_action.click(
        fn=run_medical_dashboard,
        inputs=symptom_box,
        outputs=[consensus_output, logs_output]
    )

demo.launch(inline=True, share=True)

* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://7508e5fec8dc9844be.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
